In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
np.random.seed(42)
random.seed(42)

print("Generating e-commerce sales data...")

num_transactions = 10000

categories = ['Electronics', 'Clothing', 'Home & Garden', 'Books', 'Sports']
countries = ['United States', 'United Kingdom', 'Germany', 'France', 'Canada', 
             'Australia', 'Spain', 'Italy', 'Netherlands', 'Sweden']

products = {
    'Electronics': ['Wireless Mouse', 'USB Cable', 'Keyboard', 'Webcam', 'Headphones', 'Phone Case'],
    'Clothing': ['T-Shirt', 'Jeans', 'Sweater', 'Jacket', 'Shoes', 'Hat'],
    'Home & Garden': ['Coffee Mug', 'Plant Pot', 'Candle', 'Picture Frame', 'Lamp', 'Cushion'],
    'Books': ['Fiction Novel', 'Cookbook', 'Biography', 'Self-Help Book', 'Travel Guide', 'Textbook'],
    'Sports': ['Yoga Mat', 'Water Bottle', 'Dumbbell Set', 'Running Shoes', 'Tennis Racket', 'Football']
}

price_ranges = {
    'Electronics': (15, 150),
    'Clothing': (20, 120),
    'Home & Garden': (10, 80),
    'Books': (10, 50),
    'Sports': (15, 200)
}

start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)
date_range = (end_date - start_date).days

transactions = []
customer_id = 1000

for i in range(num_transactions):
    
    category = random.choice(categories)
    product = random.choice(products[category])
    
    min_price, max_price = price_ranges[category]
    unit_price = round(random.uniform(min_price, max_price), 2)
    
    quantity = random.choices([1, 2, 3, 4, 5], weights=[50, 25, 15, 7, 3])[0]
    
    total_price = round(unit_price * quantity, 2)
    
    country = random.choice(countries)
    
    days_offset = random.randint(0, date_range)
    transaction_date = start_date + timedelta(days=days_offset)
    
    if i % 3 == 0:
        customer_id = 1000 + random.randint(0, 2000)
    else:
        customer_id = 1000 + random.randint(0, 4000)
    
    transactions.append({
        'invoice_id': f'INV-{i+10001}',
        'customer_id': f'CUST-{customer_id}',
        'transaction_date': transaction_date,
        'product': product,
        'category': category,
        'quantity': quantity,
        'unit_price': unit_price,
        'total_price': total_price,
        'country': country
    })

df = pd.DataFrame(transactions)
df = df.sort_values('transaction_date').reset_index(drop=True)

df.to_csv('ecommerce_sales_data.csv', index=False)

print(f"Generated {len(df):,} transactions")
print(f"Date range: {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Total revenue: ${df['total_price'].sum():,.2f}")
print("\nFirst 5 rows:")
print(df.head())

Generating e-commerce sales data...
Generated 10,000 transactions
Date range: 2023-01-01 to 2023-12-31
Unique customers: 3,564
Total revenue: $1,286,771.79

First 5 rows:
  invoice_id customer_id transaction_date         product       category  \
0  INV-14945   CUST-2172       2023-01-01          Candle  Home & Garden   
1  INV-18054   CUST-2476       2023-01-01   Running Shoes         Sports   
2  INV-14991   CUST-1192       2023-01-01        Yoga Mat         Sports   
3  INV-12960   CUST-2399       2023-01-01   Fiction Novel          Books   
4  INV-14535   CUST-3673       2023-01-01  Self-Help Book          Books   

   quantity  unit_price  total_price    country  
0         2       41.53        83.06      Spain  
1         1      102.18       102.18      Italy  
2         1       40.77        40.77     France  
3         1       19.31        19.31      Spain  
4         1       43.17        43.17  Australia  


In [3]:
print("Data Overview:")
print(f"Shape: {df.shape}")
print(f"\nCategories: {df['category'].unique()}")
print(f"\nRevenue by Category:")
print(df.groupby('category')['total_price'].sum().sort_values(ascending=False))
print(f"\nTop 5 Countries by Revenue:")
print(df.groupby('country')['total_price'].sum().sort_values(ascending=False).head())

Data Overview:
Shape: (10000, 9)

Categories: ['Home & Garden' 'Sports' 'Books' 'Clothing' 'Electronics']

Revenue by Category:
category
Sports           431908.26
Electronics      314312.64
Clothing         261118.80
Home & Garden    167753.41
Books            111678.68
Name: total_price, dtype: float64

Top 5 Countries by Revenue:
country
Australia         141266.91
Germany           134306.92
Italy             133795.36
Spain             132178.13
United Kingdom    127603.77
Name: total_price, dtype: float64


In [4]:
import sqlite3

conn = sqlite3.connect('ecommerce.db')

df.to_sql('sales', conn, if_exists='replace', index=False)

test_query = "SELECT COUNT(*) as total FROM sales"
result = pd.read_sql(test_query, conn)

print(f"Database created: ecommerce.db")
print(f"Table created: sales")
print(f"Total records in database: {result['total'][0]:,}")

conn.close()

Database created: ecommerce.db
Table created: sales
Total records in database: 10,000


In [5]:
conn = sqlite3.connect('ecommerce.db')

query = """
SELECT 
    category,
    COUNT(*) as num_transactions,
    SUM(total_price) as total_revenue,
    ROUND(AVG(total_price), 2) as avg_transaction_value
FROM sales
GROUP BY category
ORDER BY total_revenue DESC
"""

results = pd.read_sql(query, conn)
print("Revenue by Category:\n")
print(results)

conn.close()

Revenue by Category:

        category  num_transactions  total_revenue  avg_transaction_value
0         Sports              2078      431908.26                 207.85
1    Electronics              2007      314312.64                 156.61
2       Clothing              1959      261118.80                 133.29
3  Home & Garden              1941      167753.41                  86.43
4          Books              2015      111678.68                  55.42


In [6]:
conn = sqlite3.connect('ecommerce.db')

query = """
SELECT 
    customer_id,
    COUNT(*) as num_purchases,
    SUM(total_price) as total_spent,
    ROUND(AVG(total_price), 2) as avg_order_value
FROM sales
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10
"""

results = pd.read_sql(query, conn)
print("Top 10 Customers by Revenue:\n")
print(results)

conn.close()

Top 10 Customers by Revenue:

  customer_id  num_purchases  total_spent  avg_order_value
0   CUST-1766             10      1893.22           189.32
1   CUST-1560              7      1810.50           258.64
2   CUST-1342              9      1793.75           199.31
3   CUST-2308             11      1783.53           162.14
4   CUST-1733              8      1773.95           221.74
5   CUST-1663              7      1767.33           252.48
6   CUST-1033              5      1699.03           339.81
7   CUST-4136              4      1678.54           419.63
8   CUST-1429              7      1672.55           238.94
9   CUST-1761              5      1657.04           331.41


In [7]:
conn = sqlite3.connect('ecommerce.db')

query = """
SELECT 
    strftime('%Y-%m', transaction_date) as month,
    COUNT(*) as num_transactions,
    SUM(total_price) as monthly_revenue,
    ROUND(AVG(total_price), 2) as avg_transaction
FROM sales
GROUP BY month
ORDER BY month
"""

results = pd.read_sql(query, conn)
print("Monthly Sales Trend:\n")
print(results)

conn.close()

Monthly Sales Trend:

      month  num_transactions  monthly_revenue  avg_transaction
0   2023-01               868        115416.22           132.97
1   2023-02               768        103968.44           135.38
2   2023-03               824        109218.24           132.55
3   2023-04               850        110456.53           129.95
4   2023-05               872        108968.86           124.96
5   2023-06               781         99086.84           126.87
6   2023-07               878        108585.27           123.67
7   2023-08               874        113505.69           129.87
8   2023-09               838        109360.34           130.50
9   2023-10               813        102615.59           126.22
10  2023-11               808        106271.58           131.52
11  2023-12               826         99318.19           120.24


In [8]:
conn = sqlite3.connect('ecommerce.db')

query = """
SELECT 
    country,
    COUNT(*) as num_orders,
    SUM(total_price) as total_revenue,
    ROUND(AVG(total_price), 2) as avg_order_value
FROM sales
GROUP BY country
ORDER BY total_revenue DESC
"""

results = pd.read_sql(query, conn)
print("Revenue by Country:\n")
print(results)

conn.close()

Revenue by Country:

          country  num_orders  total_revenue  avg_order_value
0       Australia        1076      141266.91           131.29
1         Germany        1059      134306.92           126.82
2           Italy        1022      133795.36           130.92
3           Spain         997      132178.13           132.58
4  United Kingdom        1015      127603.77           125.72
5          Sweden         984      127473.83           129.55
6          France         932      124904.49           134.02
7   United States         966      122932.26           127.26
8     Netherlands         971      122027.69           125.67
9          Canada         978      120282.43           122.99


In [9]:
conn = sqlite3.connect('ecommerce.db')

query = """
SELECT 
    product,
    category,
    COUNT(*) as times_sold,
    SUM(quantity) as total_quantity,
    SUM(total_price) as total_revenue
FROM sales
GROUP BY product, category
ORDER BY total_revenue DESC
LIMIT 15
"""

results = pd.read_sql(query, conn)
print("Top 15 Products by Revenue:\n")
print(results)

conn.close()

Top 15 Products by Revenue:

           product     category  times_sold  total_quantity  total_revenue
0         Yoga Mat       Sports         363             700       79636.46
1    Running Shoes       Sports         392             737       76824.92
2         Football       Sports         342             649       69719.92
3    Tennis Racket       Sports         329             641       69537.93
4     Dumbbell Set       Sports         330             649       69349.29
5     Water Bottle       Sports         322             617       66839.74
6         Keyboard  Electronics         347             674       56305.45
7       Headphones  Electronics         357             669       56127.57
8   Wireless Mouse  Electronics         349             627       53250.14
9       Phone Case  Electronics         311             585       51121.48
10       USB Cable  Electronics         316             591       50442.75
11          Jacket     Clothing         358             688       47892

In [10]:
conn = sqlite3.connect('ecommerce.db')

query = """
SELECT 
    num_purchases,
    COUNT(*) as num_customers,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(DISTINCT customer_id) FROM sales), 2) as percentage
FROM (
    SELECT customer_id, COUNT(*) as num_purchases
    FROM sales
    GROUP BY customer_id
)
GROUP BY num_purchases
ORDER BY num_purchases
"""

results = pd.read_sql(query, conn)
print("Customer Purchase Frequency Distribution:\n")
print(results)

conn.close()

Customer Purchase Frequency Distribution:

    num_purchases  num_customers  percentage
0               1            860       24.13
1               2            939       26.35
2               3            735       20.62
3               4            504       14.14
4               5            270        7.58
5               6            150        4.21
6               7             69        1.94
7               8             28        0.79
8               9              7        0.20
9              10              1        0.03
10             11              1        0.03


In [11]:
print("Calculating RFM scores...")

reference_date = df['transaction_date'].max() + timedelta(days=1)

rfm = df.groupby('customer_id').agg({
    'transaction_date': lambda x: (reference_date - x.max()).days,
    'invoice_id': 'count',
    'total_price': 'sum'
})

rfm.columns = ['recency', 'frequency', 'monetary']

rfm['recency_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop')
rfm['frequency_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
rfm['monetary_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')

rfm['rfm_score'] = rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str) + rfm['monetary_score'].astype(str)

print(f"RFM scores calculated for {len(rfm)} customers")
print("\nSample RFM data:")
print(rfm.head(10))

Calculating RFM scores...
RFM scores calculated for 3564 customers

Sample RFM data:
             recency  frequency  monetary recency_score frequency_score  \
customer_id                                                               
CUST-1000         13          4    449.70             5               4   
CUST-1001        189          1     60.16             2               1   
CUST-1002         27          4    562.61             5               4   
CUST-1003        116          2    220.34             3               2   
CUST-1004        124          4    202.96             2               4   
CUST-1005          7          4    389.37             5               4   
CUST-1006        221          3    413.87             1               3   
CUST-1007        138          1     70.42             2               1   
CUST-1008         22          6    413.62             5               5   
CUST-1009         23          4    272.26             5               4   

            mo

In [12]:
def segment_customers(df):
    if df['recency_score'] >= 4 and df['frequency_score'] >= 4:
        return 'Champions'
    elif df['recency_score'] >= 3 and df['frequency_score'] >= 3:
        return 'Loyal Customers'
    elif df['recency_score'] >= 4 and df['frequency_score'] <= 2:
        return 'Potential Loyalists'
    elif df['recency_score'] >= 3 and df['frequency_score'] <= 2:
        return 'New Customers'
    elif df['recency_score'] <= 2 and df['frequency_score'] >= 3:
        return 'At Risk'
    elif df['recency_score'] <= 2 and df['frequency_score'] <= 2:
        return 'Lost Customers'
    else:
        return 'Others'

rfm['segment'] = rfm.apply(segment_customers, axis=1)

print("Customer Segmentation:")
print(rfm['segment'].value_counts())

print("\nSegment Summary:")
segment_summary = rfm.groupby('segment').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).round(2)
print(segment_summary)

Customer Segmentation:
segment
Lost Customers         839
Champions              815
Loyal Customers        743
At Risk                580
Potential Loyalists    352
New Customers          235
Name: count, dtype: int64

Segment Summary:
                     recency  frequency  monetary
segment                                          
At Risk               177.18       3.15    402.13
Champions              30.69       4.61    597.78
Lost Customers        232.42       1.30    161.82
Loyal Customers        70.49       3.27    430.57
New Customers          95.48       1.51    193.74
Potential Loyalists    34.95       1.56    185.03


In [13]:
rfm_output = rfm.reset_index()
rfm_output.to_csv('customer_segments.csv', index=False)

print("Customer segments saved to: customer_segments.csv")
print(f"\nTotal customers analyzed: {len(rfm_output)}")
print("\nSegment distribution:")
print(rfm_output['segment'].value_counts())

Customer segments saved to: customer_segments.csv

Total customers analyzed: 3564

Segment distribution:
segment
Lost Customers         839
Champions              815
Loyal Customers        743
At Risk                580
Potential Loyalists    352
New Customers          235
Name: count, dtype: int64


In [14]:
sales_by_category = df.groupby('category').agg({
    'total_price': 'sum',
    'invoice_id': 'count'
}).reset_index()

sales_by_category.columns = ['category', 'total_revenue', 'num_transactions']
sales_by_category = sales_by_category.sort_values('total_revenue', ascending=False)
sales_by_category.to_csv('sales_by_category.csv', index=False)

print("Sales by category exported")
print(sales_by_category)

Sales by category exported
        category  total_revenue  num_transactions
4         Sports      431908.26              2078
2    Electronics      314312.64              2007
1       Clothing      261118.80              1959
3  Home & Garden      167753.41              1941
0          Books      111678.68              2015


In [15]:
df['year_month'] = pd.to_datetime(df['transaction_date']).dt.to_period('M').astype(str)

monthly_sales = df.groupby('year_month').agg({
    'total_price': 'sum',
    'invoice_id': 'count'
}).reset_index()

monthly_sales.columns = ['month', 'total_revenue', 'num_transactions']
monthly_sales.to_csv('monthly_sales.csv', index=False)

print("Monthly sales exported")
print(monthly_sales)

Monthly sales exported
      month  total_revenue  num_transactions
0   2023-01      115416.22               868
1   2023-02      103968.44               768
2   2023-03      109218.24               824
3   2023-04      110456.53               850
4   2023-05      108968.86               872
5   2023-06       99086.84               781
6   2023-07      108585.27               878
7   2023-08      113505.69               874
8   2023-09      109360.34               838
9   2023-10      102615.59               813
10  2023-11      106271.58               808
11  2023-12       99318.19               826


In [16]:
sales_by_country = df.groupby('country').agg({
    'total_price': 'sum',
    'invoice_id': 'count'
}).reset_index()

sales_by_country.columns = ['country', 'total_revenue', 'num_transactions']
sales_by_country = sales_by_country.sort_values('total_revenue', ascending=False)
sales_by_country.to_csv('sales_by_country.csv', index=False)

print("Sales by country exported")
print(sales_by_country)

Sales by country exported
          country  total_revenue  num_transactions
0       Australia      141266.91              1076
3         Germany      134306.92              1059
4           Italy      133795.36              1022
6           Spain      132178.13               997
8  United Kingdom      127603.77              1015
7          Sweden      127473.83               984
2          France      124904.49               932
9   United States      122932.26               966
5     Netherlands      122027.69               971
1          Canada      120282.43               978


In [17]:
top_products = df.groupby(['product', 'category']).agg({
    'quantity': 'sum',
    'total_price': 'sum',
    'invoice_id': 'count'
}).reset_index()

top_products.columns = ['product', 'category', 'total_quantity', 'total_revenue', 'times_sold']
top_products = top_products.sort_values('total_revenue', ascending=False).head(20)
top_products.to_csv('top_products.csv', index=False)

print("Top 20 products exported")
print(top_products)

Top 20 products exported
           product       category  total_quantity  total_revenue  times_sold
29        Yoga Mat         Sports             700       79636.46         363
17   Running Shoes         Sports             737       76824.92         392
7         Football         Sports             649       69719.92         342
22   Tennis Racket         Sports             641       69537.93         329
5     Dumbbell Set         Sports             649       69349.29         330
26    Water Bottle         Sports             617       66839.74         322
12        Keyboard    Electronics             674       56305.45         347
9       Headphones    Electronics             669       56127.57         357
28  Wireless Mouse    Electronics             627       53250.14         349
14      Phone Case    Electronics             585       51121.48         311
25       USB Cable    Electronics             591       50442.75         316
10          Jacket       Clothing             688  

In [18]:
print("=" * 60)
print("PROJECT 1: E-COMMERCE ANALYSIS - DATA EXPORT COMPLETE")
print("=" * 60)

print("\nFiles created:")
print("1. ecommerce_sales_data.csv - All transactions")
print("2. ecommerce.db - SQLite database")
print("3. customer_segments.csv - RFM analysis results")
print("4. sales_by_category.csv - Revenue by product category")
print("5. monthly_sales.csv - Monthly sales trends")
print("6. sales_by_country.csv - Revenue by country")
print("7. top_products.csv - Best-selling products")

print("\nData Summary:")
print(f"Total transactions: {len(df):,}")
print(f"Total revenue: ${df['total_price'].sum():,.2f}")
print(f"Total customers: {df['customer_id'].nunique():,}")
print(f"Date range: {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")

print("\nCustomer Segments:")
print(rfm_output['segment'].value_counts())

print("\nTop 3 Revenue Categories:")
print(sales_by_category.head(3))

print("\n" + "=" * 60)
print("All data ready for Tableau visualization!")
print("=" * 60)

PROJECT 1: E-COMMERCE ANALYSIS - DATA EXPORT COMPLETE

Files created:
1. ecommerce_sales_data.csv - All transactions
2. ecommerce.db - SQLite database
3. customer_segments.csv - RFM analysis results
4. sales_by_category.csv - Revenue by product category
5. monthly_sales.csv - Monthly sales trends
6. sales_by_country.csv - Revenue by country
7. top_products.csv - Best-selling products

Data Summary:
Total transactions: 10,000
Total revenue: $1,286,771.79
Total customers: 3,564
Date range: 2023-01-01 to 2023-12-31

Customer Segments:
segment
Lost Customers         839
Champions              815
Loyal Customers        743
At Risk                580
Potential Loyalists    352
New Customers          235
Name: count, dtype: int64

Top 3 Revenue Categories:
      category  total_revenue  num_transactions
4       Sports      431908.26              2078
2  Electronics      314312.64              2007
1     Clothing      261118.80              1959

All data ready for Tableau visualization!
